In [1]:
import polars as pl
import datetime as dt
import random

from typing import Literal

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
now = dt.datetime(2026, 1, 1)
expiry = dt.datetime(2026, 1, 2)
symbol = 'BTC'

### Value block data structure

In [3]:
def annualize(value, seconds):
    return value * (365.25 * 24 * 60 * 60) / seconds

def deannualize(value, seconds):
    return value / (365.25 * 24 * 60 * 60) * seconds

class ValueBlock():
    def __init__(
        #-----Static parameters-----#
        # defaults are for simple base vol pricing
        self,
        block_name:str,
        annualized:bool=True,
        size_type:Literal['fixed', 'relative']='fixed', # Fixed size or size relative to market implied
        aggregation_logic:Literal['average', 'offset']='average', # i.e., fundamental/realised (average all fundamental valuations) or implied (sum all expected price moves)
        temporal_position:Literal['static', 'shifting']='shifting', # if shifting, start timestamp is always now
        decay_end_size_mult:float=1.0, # can only be non-zero for annualized; decay start size is calculated by size, since total variance contribution is fixed
        decay_rate_prop_per_min:float=0.0, # speed of total variance decay
        decay_profile:Literal['linear']='linear', # decay is in total variance space, so actually looks like uniform block on variance x timestamp graph
        start_timestamp:dt.datetime=None, # only applicable for static streams
        
        #----- Dynamic parameters -----#
        var_fair_ratio:float=1.0,
    ):
        self.block_name = block_name
        self.annualized = annualized
        self.size_type = size_type
        self.aggregation_logic = aggregation_logic
        self.temporal_position = temporal_position
        self.decay_end_size_mult = decay_end_size_mult
        self.decay_rate_prop_per_min = decay_rate_prop_per_min
        self.decay_profile = decay_profile
        self.start_timestamp = start_timestamp
        
        self.var_fair_ratio = var_fair_ratio
        
        self._check_attributes()
        
    def _check_attributes(self):
        # Choice variables
        assert self.size_type in ['fixed', 'relative'], "size_type must be 'fixed' or 'relative'"
        assert self.aggregation_logic in ['average', 'offset'], "aggregation_logic must be 'average' or 'offset'"
        assert self.temporal_position in ['static', 'shifting'], "temporal_position must be 'static' or 'shifting'"
        assert self.decay_profile in ['linear'], "decay_profile must be 'linear'"
        
        # Numerical variables
        assert self.decay_end_size_mult >= 0, "decay_end_size_mult must be non-negative"
        assert self.decay_rate_prop_per_min >= 0, "decay_rate_prop_per_min must be non-negative"
        
        # Special conditions
        if self.size_type == 'relative': # not necessary, can adapt for total type streams
            assert self.annualized, " must be annualized for relative size"
        if self.decay_end_size_mult != 0 and not self.annualized:
            raise ValueError("decay_end_size_mult is only applicable for annualized streams")
        if self.temporal_position == "static" and self.start_timestamp is None:
            raise ValueError("start_timestamp must be provided for static streams")
        
        
    def _get_end_timestamp(self, start_ts:dt.datetime, expiry:dt.datetime):
        if self.decay_end_size_mult == 1 or self.decay_rate_prop_per_min == 0:
            return expiry
        else:
            return start_ts + dt.timedelta(minutes=1 / self.decay_rate_prop_per_min)
        
        
    def _get_total_value(self, stream_value:float, market_value:float, start_ts:dt.datetime, end_ts:dt.datetime):
        if self.annualized: # Relative pricing not applicable for total type stream, since we're assuming market_value is always annualised
            annualized_value = stream_value if self.size_type=='fixed' else stream_value - market_value
            return deannualize(annualized_value, (end_ts - start_ts).total_seconds())
        else:
            return stream_value
        
        
    def _get_start_annualized_value(self, total_value:float, expiry:dt.datetime, start_ts:dt.datetime, end_ts:dt.datetime, end_annualized_value:float):
        start_to_expiry_secs = (expiry - start_ts).total_seconds()
        start_to_end_secs = (end_ts - start_ts).total_seconds()
        annualized_value = annualize(total_value, start_to_end_secs)
            
        if self.annualized:
            start_to_end_secs = min(start_to_expiry_secs, start_to_end_secs)
            # Stream Value = Prop. time between start and end * (Average of Start and End Value, assuming linear) + (1-Prop.) * End Value
            # v = p * (s + e) / 2 + (1 - p) * e
            # ==> s = (2 / p) * (v - (1 - p) * e) - e
            p = start_to_end_secs / start_to_expiry_secs
            return (2 / p) * (annualized_value - (1 - p) * end_annualized_value) - end_annualized_value
        else:
            return annualized_value
    
    def _fair_annualized_col(self, column_prefix:str=''):
        return f'{column_prefix}{self.block_name}_fair_annualized'
    
    def _fair_col(self, column_prefix:str=''):
        return f'{column_prefix}{self.block_name}_fair'
    
    def _var_col(self, column_prefix:str=''):
        return f'{column_prefix}{self.block_name}_var'
    
    def calculate_fair(self, now:dt.datetime, stream_value:float, market_value:float, future_df:pl.DataFrame, expiry:dt.datetime, column_prefix:str=''):
        fair_annualized_col = self._fair_annualized_col(column_prefix)
        fair_col = self._fair_col(column_prefix)
        
        # Determine timestamp range
        start_ts = now if self.temporal_position == "shifting" else self.start_timestamp
        end_ts = self._get_end_timestamp(start_ts, expiry)
        
        # Calculate total value (e.g., total variance)
        total_value = self._get_total_value(stream_value, market_value, start_ts, end_ts)
        
        # Calculate start and end instantaneous annualised variance
        end_annualized_value = annualize(total_value, (end_ts - start_ts).total_seconds()) * self.decay_end_size_mult # doesn't matter that stream_value isn't annualised for 'total' type streams, since mult will always be 0
        start_annualized_value = self._get_start_annualized_value(total_value, expiry, start_ts, end_ts, end_annualized_value)
        
        # Add column to future_df for fair value
        out_df = future_df.with_columns(
            (
                pl.when(pl.col('timestamp') < start_ts)
                .then(0)
                .when(pl.col('timestamp') > end_ts)
                .then(end_annualized_value)
                .when(self.annualized)
                .then(
                    # only applicable for linear
                    start_annualized_value + (end_annualized_value - start_annualized_value) * (pl.col('timestamp') - start_ts) / (end_ts - start_ts)
                )
                .otherwise(start_annualized_value) # only applicable for linear
            ).alias(fair_annualized_col)
        ).with_columns(
            (pl.col(fair_annualized_col) * pl.col('dtte')).alias(fair_col)
        )
        return out_df
    
    
    def calculate_variance(self, future_df:pl.DataFrame, column_prefix:str=''):
        fair_col = self._fair_col(column_prefix)
        var_col = self._var_col(column_prefix)
        return future_df.with_columns(
            (pl.col(fair_col).abs() * self.var_fair_ratio).alias(var_col)
        )
    
    
    def calculate_fair_and_variance(self, now:dt.datetime, stream_value:float, market_value:float, future_df:pl.DataFrame, expiry:dt.datetime, column_prefix:str=''):
        out_df = self.calculate_fair(now, stream_value, market_value, future_df, expiry, column_prefix=column_prefix)
        out_df = self.calculate_variance(out_df, column_prefix=column_prefix)
        return out_df

### Data Stream data structure

In [4]:
def general_raw_to_target_transformation(raw_value:float, scale:float, offset:float, exponent:float) -> float:
    return (scale * raw_value + offset)**exponent

class DataStream():
    def __init__(
            self,
            stream_name:str,
            snapshot:pl.DataFrame, 
            key_cols:list[str],
            scale:float=None,
            offset:float=None,
            exponent:float=None,
            annualized:bool=None,
            
            # Value blocks
            size_type:Literal['fixed', 'relative']='fixed', # Fixed size or size relative to market implied
            aggregation_logic:Literal['average', 'offset']='average', # i.e., fundamental/realised (average all fundamental valuations) or implied (sum all expected price moves)
            temporal_position:Literal['static', 'shifting']='shifting', # if shifting, start timestamp is always now
            decay_end_size_mult:float=1.0, # can only be non-zero for annualized; decay start size is calculated by size, since total variance contribution is fixed
            decay_rate_prop_per_min:float=0.0, # speed of total variance decay
            decay_profile:Literal['linear']='linear', # decay is in total variance space, so actually looks like uniform block on variance x timestamp graph
            var_fair_ratio:float=1.0,
        ):
        self.stream_name = stream_name
        self.snapshot = snapshot
        self.key_cols = key_cols
        self.extra_key_cols = [k for k in key_cols if k not in ('symbol', 'expiry')]
        
        # To be determined by LLM
        self.scale  = scale
        self.offset = offset
        self.exponent = exponent
        self.annualized = annualized
        
        # Value blocks parameters
        self.size_type = size_type
        self.aggregation_logic = aggregation_logic
        self.temporal_position = temporal_position
        self.decay_end_size_mult = decay_end_size_mult
        self.decay_rate_prop_per_min = decay_rate_prop_per_min
        self.decay_profile = decay_profile
        self.var_fair_ratio = var_fair_ratio
        
        self.update_snapshot(self.snapshot)
        
    def update_snapshot(self, snapshot:pl.DataFrame):
        self.snapshot = snapshot
        if 'start_timestamp' not in self.snapshot.columns:
            self.snapshot = self.snapshot.with_columns(pl.lit(None).cast(pl.Datetime('us')).alias('start_timestamp'))
        self.get_latest_snapshot()
        self.transform_raw_to_target()
        self.get_value_blocks()
        
    def get_latest_snapshot(self):
        self.snapshot = self.snapshot.sort('timestamp').group_by(self.key_cols).agg(pl.all().last())
        
    def transform_raw_to_target(self):
        self.snapshot = self.snapshot.with_columns(
            pl.col('raw_value').map_elements(lambda x: general_raw_to_target_transformation(x, self.scale, self.offset, self.exponent)).alias('target_value'),
        )
    
    def add_market_value(self, space_market_pricing:pl.DataFrame):
        self.snapshot = self.snapshot.join(space_market_pricing, on='space_id', how='left')
        self.snapshot = self.snapshot.with_columns(
            pl.col('market_value').map_elements(lambda x: general_raw_to_target_transformation(x, self.scale, self.offset, self.exponent)).alias('target_market_value'),
        )
        
    def _make_block_name(self, row:dict) -> str:
        parts = [self.stream_name]
        for k in self.extra_key_cols:
            parts.append(str(row[k]))
        return "_".join(parts)
        
    def get_value_blocks(self):
        block_map = {}
        for keys, group in self.snapshot.group_by(self.key_cols):
            key_tuple = keys if isinstance(keys, tuple) else (keys,)
            row = dict(zip(self.key_cols, key_tuple))
            block_name = self._make_block_name(row)
            
            # Read start_timestamp from the snapshot row (per-row parameter)
            row_data = group.row(0, named=True)
            start_timestamp = row_data.get('start_timestamp', None)
            
            block_map[key_tuple] = ValueBlock(
                block_name=block_name,
                annualized=self.annualized,
                size_type=self.size_type,
                aggregation_logic=self.aggregation_logic,
                temporal_position=self.temporal_position,
                decay_end_size_mult=self.decay_end_size_mult,
                decay_rate_prop_per_min=self.decay_rate_prop_per_min,
                decay_profile=self.decay_profile,
                start_timestamp=start_timestamp,
                var_fair_ratio=self.var_fair_ratio,
            )
    
        self.snapshot = self.snapshot.with_columns(
            pl.struct(self.key_cols)
            .map_elements(lambda row: block_map[tuple(row.values())], return_dtype=pl.Object)
            .alias("value_block")
        )

### Stream initialisation + Conversion to target space

In [5]:
# Snapshots are sent per data stream

mock_timestamp = now

RVStream = DataStream(
    'rv',
    pl.DataFrame({
        'timestamp' : mock_timestamp,
        'symbol' : symbol,
        'expiry' : expiry,
        'raw_value' : random.random(),
    }),
    ['symbol', 'expiry'],
    scale=1, offset=0, exponent=2,
    annualized=True
)

MeanIVStream = DataStream(
    'mean_iv',
    pl.DataFrame({
        'timestamp' : mock_timestamp,
        'symbol' : symbol,
        'expiry' : expiry,
        'raw_value' : random.random(),
    }),
    ['symbol', 'expiry'],
    scale=1, offset=0, exponent=2,
    annualized=True, aggregation_logic='offset', size_type='relative',
    decay_end_size_mult=0.0, decay_rate_prop_per_min=0.01, decay_profile='linear',
    var_fair_ratio=2.0 # assume we're less confidence in this -- less confident = higher variance
)

num_events = 5
event_start_timestamps = [now + dt.timedelta(hours=i * 4) for i in range(num_events)]
EventsStream = DataStream(
    'events',
    pl.DataFrame({
        'timestamp' : [mock_timestamp for _ in range(num_events)],
        'symbol' : symbol,
        'expiry' : expiry,
        'event_id' : [f'event_{i}' for i in range(num_events)],
        'raw_value' : [random.random() * 5 for _ in range(num_events)],
        'start_timestamp' : event_start_timestamps,
    }),
    ['symbol', 'expiry', 'event_id'],
    scale=1/100, offset=0, exponent=2,
    annualized=False, aggregation_logic='offset', size_type='fixed',
    temporal_position='static',
    decay_end_size_mult=0.0, decay_rate_prop_per_min=0.01, decay_profile='linear',
    var_fair_ratio=3.0
)

data_streams = [RVStream, MeanIVStream, EventsStream]

for stream in data_streams:
    display(stream.snapshot)

symbol,expiry,timestamp,raw_value,start_timestamp,target_value,value_block
str,datetime[μs],datetime[μs],f64,datetime[μs],f64,object
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:00:00,0.102042,null,0.010412,<__main__.ValueBlock object at 0x109cc2510>


symbol,expiry,timestamp,raw_value,start_timestamp,target_value,value_block
str,datetime[μs],datetime[μs],f64,datetime[μs],f64,object
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:00:00,0.36879,null,0.136006,<__main__.ValueBlock object at 0x109156d50>


symbol,expiry,event_id,timestamp,raw_value,start_timestamp,target_value,value_block
str,datetime[μs],str,datetime[μs],f64,datetime[μs],f64,object
"""BTC""",2026-01-02 00:00:00,"""event_2""",2026-01-01 00:00:00,1.843572,2026-01-01 08:00:00,0.00034,<__main__.ValueBlock object at 0x109dc98c0>
"""BTC""",2026-01-02 00:00:00,"""event_3""",2026-01-01 00:00:00,2.082659,2026-01-01 12:00:00,0.000434,<__main__.ValueBlock object at 0x109dd9130>
"""BTC""",2026-01-02 00:00:00,"""event_1""",2026-01-01 00:00:00,2.184375,2026-01-01 04:00:00,0.000477,<__main__.ValueBlock object at 0x109157390>
"""BTC""",2026-01-02 00:00:00,"""event_0""",2026-01-01 00:00:00,1.683827,2026-01-01 00:00:00,0.000284,<__main__.ValueBlock object at 0x109dc10f0>
"""BTC""",2026-01-02 00:00:00,"""event_4""",2026-01-01 00:00:00,0.775306,2026-01-01 16:00:00,0.00006,<__main__.ValueBlock object at 0x109dc1220>


#### Space assignment based on temporal relationship

In [6]:
def assign_spaces(data_streams: list[DataStream]) -> pl.DataFrame:
    """Assign a space_id to each snapshot row based on temporal_position and start_timestamp.
    
    - All rows in 'shifting' streams share one space.
    - Each row in 'static' streams gets a unique space per unique start_timestamp.
    
    Mutates each stream's snapshot by adding a 'space_id' column.
    Returns a summary DataFrame with one row per unique (stream, key) combination.
    """
    static_space_map: dict[dt.datetime, str] = {}
    summary_rows = []

    for stream in data_streams:
        tp = stream.temporal_position

        if tp == 'shifting':
            stream.snapshot = stream.snapshot.with_columns(pl.lit('shifting').alias('space_id'))
        else:
            # Build space_id per row from start_timestamp
            space_ids = []
            for st in stream.snapshot['start_timestamp'].to_list():
                if st not in static_space_map:
                    static_space_map[st] = f'static_{st.strftime("%Y%m%d_%H%M%S")}'
                space_ids.append(static_space_map[st])
            stream.snapshot = stream.snapshot.with_columns(pl.Series('space_id', space_ids, dtype=pl.Utf8))

        # Build summary rows
        for row in stream.snapshot.iter_rows(named=True):
            summary_rows.append({
                'stream_name': stream.stream_name,
                'temporal_position': tp,
                'start_timestamp': row.get('start_timestamp'),
                'space_id': row['space_id'],
            })

    return pl.DataFrame(summary_rows, schema={
        'stream_name': pl.Utf8,
        'temporal_position': pl.Utf8,
        'start_timestamp': pl.Datetime('us'),
        'space_id': pl.Utf8,
    })

space_df = assign_spaces(data_streams)
display(space_df)

stream_name,temporal_position,start_timestamp,space_id
str,str,datetime[μs],str
"""rv""","""shifting""",null,"""shifting"""
"""mean_iv""","""shifting""",null,"""shifting"""
"""events""","""static""",2026-01-01 08:00:00,"""static_20260101_080000"""
"""events""","""static""",2026-01-01 12:00:00,"""static_20260101_120000"""
"""events""","""static""",2026-01-01 04:00:00,"""static_20260101_040000"""
"""events""","""static""",2026-01-01 00:00:00,"""static_20260101_000000"""
"""events""","""static""",2026-01-01 16:00:00,"""static_20260101_160000"""


In [7]:
# Mock market implied pricing per space (user provides real data here in practice)
unique_spaces = space_df['space_id'].unique().to_list()
space_market_pricing = pl.DataFrame({
    'space_id': unique_spaces,
    'market_value': [round(random.uniform(0.1, 1.0), 4) for _ in unique_spaces],
})
display(space_market_pricing)

for stream in data_streams:
    stream.add_market_value(space_market_pricing)
    display(stream.snapshot)

space_id,market_value
str,f64
"""static_20260101_080000""",0.9363
"""shifting""",0.9307
"""static_20260101_040000""",0.2381
"""static_20260101_120000""",0.6932
"""static_20260101_000000""",0.144
"""static_20260101_160000""",0.7947


symbol,expiry,timestamp,raw_value,start_timestamp,target_value,value_block,space_id,market_value,target_market_value
str,datetime[μs],datetime[μs],f64,datetime[μs],f64,object,str,f64,f64
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:00:00,0.102042,null,0.010412,<__main__.ValueBlock object at 0x109cc2510>,"""shifting""",0.9307,0.866202


symbol,expiry,timestamp,raw_value,start_timestamp,target_value,value_block,space_id,market_value,target_market_value
str,datetime[μs],datetime[μs],f64,datetime[μs],f64,object,str,f64,f64
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:00:00,0.36879,null,0.136006,<__main__.ValueBlock object at 0x109156d50>,"""shifting""",0.9307,0.866202


symbol,expiry,event_id,timestamp,raw_value,start_timestamp,target_value,value_block,space_id,market_value,target_market_value
str,datetime[μs],str,datetime[μs],f64,datetime[μs],f64,object,str,f64,f64
"""BTC""",2026-01-02 00:00:00,"""event_2""",2026-01-01 00:00:00,1.843572,2026-01-01 08:00:00,0.00034,<__main__.ValueBlock object at 0x109dc98c0>,"""static_20260101_080000""",0.9363,0.000088
"""BTC""",2026-01-02 00:00:00,"""event_3""",2026-01-01 00:00:00,2.082659,2026-01-01 12:00:00,0.000434,<__main__.ValueBlock object at 0x109dd9130>,"""static_20260101_120000""",0.6932,0.000048
"""BTC""",2026-01-02 00:00:00,"""event_1""",2026-01-01 00:00:00,2.184375,2026-01-01 04:00:00,0.000477,<__main__.ValueBlock object at 0x109157390>,"""static_20260101_040000""",0.2381,0.000006
"""BTC""",2026-01-02 00:00:00,"""event_0""",2026-01-01 00:00:00,1.683827,2026-01-01 00:00:00,0.000284,<__main__.ValueBlock object at 0x109dc10f0>,"""static_20260101_000000""",0.144,0.000002
"""BTC""",2026-01-02 00:00:00,"""event_4""",2026-01-01 00:00:00,0.775306,2026-01-01 16:00:00,0.00006,<__main__.ValueBlock object at 0x109dc1220>,"""static_20260101_160000""",0.7947,0.000063


### Fair value / variance conversion

In [8]:
future_df = pl.DataFrame({
    'timestamp' : pl.datetime_range(
        start=now,
        end=expiry,
        interval='1s',
        eager=True
    ),
    'symbol' : symbol,
    'expiry' : expiry,
}).with_columns(
    dtte=-pl.col('timestamp').diff(-1).dt.total_seconds() / (60 * 60 * 24 * 365.25),
)
future_df

timestamp,symbol,expiry,dtte
datetime[μs],str,datetime[μs],f64
2026-01-01 00:00:00,"""BTC""",2026-01-02 00:00:00,3.1688e-8
2026-01-01 00:00:01,"""BTC""",2026-01-02 00:00:00,3.1688e-8
2026-01-01 00:00:02,"""BTC""",2026-01-02 00:00:00,3.1688e-8
2026-01-01 00:00:03,"""BTC""",2026-01-02 00:00:00,3.1688e-8
2026-01-01 00:00:04,"""BTC""",2026-01-02 00:00:00,3.1688e-8
…,…,…,…
2026-01-01 23:59:56,"""BTC""",2026-01-02 00:00:00,3.1688e-8
2026-01-01 23:59:57,"""BTC""",2026-01-02 00:00:00,3.1688e-8
2026-01-01 23:59:58,"""BTC""",2026-01-02 00:00:00,3.1688e-8


In [9]:
# Enrich future_df with fair/var from each DataStream's value blocks, then aggregate per space

def sum_exprs(exprs: list[pl.Expr]) -> pl.Expr:
    total = pl.lit(0.0)
    for expr in exprs:
        total = total + expr
    return total

enriched_parts = []
for (sym, exp), sub_df in future_df.group_by('symbol', 'expiry'):
    now_ts = sub_df['timestamp'].min()
    space_specs = {}

    for stream in data_streams:
        matches = stream.snapshot.filter(
            (pl.col('symbol') == sym) & (pl.col('expiry') == exp)
        )

        for row in matches.iter_rows(named=True):
            space_id = row.get('space_id')
            block = row.get('value_block')
            target_value = row.get('target_value')
            target_market_value = row.get('target_market_value')

            if space_id is None:
                raise ValueError(f"Missing space_id for stream {stream.stream_name}")
            if block is None:
                raise ValueError(f"Missing value_block for stream {stream.stream_name}, space {space_id}")
            if target_value is None:
                raise ValueError(f"Missing target_value for stream {stream.stream_name}, space {space_id}")
            if target_market_value is None:
                raise ValueError(f"Missing target_market_value for stream {stream.stream_name}, space {space_id}")

            if space_id not in space_specs:
                space_specs[space_id] = {
                    'target_market_value': target_market_value,
                    'average_blocks': [],
                    'offset_blocks': [],
                    'all_blocks': [],
                }
            elif space_specs[space_id]['target_market_value'] != target_market_value:
                raise ValueError(
                    f"Inconsistent target_market_value in space {space_id}: "
                    f"{space_specs[space_id]['target_market_value']} vs {target_market_value}"
                )

            block_spec = {
                'block': block,
                'target_value': target_value,
                'target_market_value': target_market_value,
            }

            if block.aggregation_logic == 'average':
                space_specs[space_id]['average_blocks'].append(block_spec)
            else:
                space_specs[space_id]['offset_blocks'].append(block_spec)

            space_specs[space_id]['all_blocks'].append(block_spec)

    for space_spec in space_specs.values():
        for block_spec in space_spec['all_blocks']:
            block = block_spec['block']
            sub_df = block.calculate_fair_and_variance(
                now=now_ts,
                stream_value=block_spec['target_value'],
                market_value=block_spec['target_market_value'],
                future_df=sub_df,
                expiry=exp,
            )
            sub_df = block.calculate_fair(
                now=now_ts,
                stream_value=block_spec['target_market_value'],
                market_value=block_spec['target_market_value'],
                future_df=sub_df,
                expiry=exp,
                column_prefix='market_',
            )

    space_edge_exprs = []
    space_var_exprs = []

    for space_spec in space_specs.values():
        average_fair_exprs = [
            pl.col(block_spec['block']._fair_col())
            for block_spec in space_spec['average_blocks']
        ]
        offset_fair_exprs = [
            pl.col(block_spec['block']._fair_col())
            for block_spec in space_spec['offset_blocks']
        ]
        average_market_fair_exprs = [
            pl.col(block_spec['block']._fair_col('market_'))
            for block_spec in space_spec['average_blocks']
        ]
        offset_market_fair_exprs = [
            pl.col(block_spec['block']._fair_col('market_'))
            for block_spec in space_spec['offset_blocks']
        ]
        var_exprs = [
            pl.col(block_spec['block']._var_col())
            for block_spec in space_spec['all_blocks']
        ]

        space_average_fair = (
            sum_exprs(average_fair_exprs) / len(average_fair_exprs)
            if average_fair_exprs else pl.lit(0.0)
        )
        space_offset_fair = sum_exprs(offset_fair_exprs)
        space_market_average_fair = (
            sum_exprs(average_market_fair_exprs) / len(average_market_fair_exprs)
            if average_market_fair_exprs else pl.lit(0.0)
        )
        space_market_offset_fair = sum_exprs(offset_market_fair_exprs)
        space_fair = space_average_fair + space_offset_fair
        space_market_fair = space_market_average_fair + space_market_offset_fair
        space_edge = space_fair - space_market_fair
        space_var = sum_exprs(var_exprs)

        space_edge_exprs.append(space_edge)
        space_var_exprs.append(space_var)

    sub_df = sub_df.with_columns(
        edge=sum_exprs(space_edge_exprs),
        var=sum_exprs(space_var_exprs),
    )

    enriched_parts.append(sub_df)

fair_var_df = pl.concat(enriched_parts)
display(fair_var_df)

timestamp,symbol,expiry,dtte,rv_fair_annualized,rv_fair,rv_var,market_rv_fair_annualized,market_rv_fair,mean_iv_fair_annualized,mean_iv_fair,mean_iv_var,market_mean_iv_fair_annualized,market_mean_iv_fair,events_event_2_fair_annualized,events_event_2_fair,events_event_2_var,market_events_event_2_fair_annualized,market_events_event_2_fair,events_event_3_fair_annualized,events_event_3_fair,events_event_3_var,market_events_event_3_fair_annualized,market_events_event_3_fair,events_event_1_fair_annualized,events_event_1_fair,events_event_1_var,market_events_event_1_fair_annualized,market_events_event_1_fair,events_event_0_fair_annualized,events_event_0_fair,events_event_0_var,market_events_event_0_fair_annualized,market_events_event_0_fair,events_event_4_fair_annualized,events_event_4_fair,events_event_4_var,market_events_event_4_fair_annualized,market_events_event_4_fair,edge,var
datetime[μs],str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-01-01 00:00:00,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-21.029648,-6.6639e-7,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.49124,4.7255e-8,1.4176e-7,0.010906,3.4560e-10,0.0,0.0,0.0,0.0,0.0,-6.4660e-7,0.000001
2026-01-01 00:00:01,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-21.026143,-6.6628e-7,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.49124,4.7255e-8,1.4176e-7,0.010906,3.4560e-10,0.0,0.0,0.0,0.0,0.0,-6.4649e-7,0.000001
2026-01-01 00:00:02,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-21.022638,-6.6617e-7,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.49124,4.7255e-8,1.4176e-7,0.010906,3.4560e-10,0.0,0.0,0.0,0.0,0.0,-6.4638e-7,0.000001
2026-01-01 00:00:03,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-21.019133,-6.6606e-7,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.49124,4.7255e-8,1.4176e-7,0.010906,3.4560e-10,0.0,0.0,0.0,0.0,0.0,-6.4627e-7,0.000001
2026-01-01 00:00:04,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-21.015628,-6.6595e-7,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.49124,4.7255e-8,1.4176e-7,0.010906,3.4560e-10,0.0,0.0,0.0,0.0,0.0,-6.4615e-7,0.000001
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-01-01 23:59:56,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.7118e-8,3.2995e-10
2026-01-01 23:59:57,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.7118e-8,3.2995e-10
2026-01-01 23:59:58,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.7118e-8,3.2995e-10


### Desired position calculation

In [10]:
smoothing_hl_secs = 60 * 15
bankroll = 100_000

In [11]:
desired_position_df = fair_var_df.with_columns(
    raw_desired_position = pl.col('edge') * bankroll / pl.col('var')
).with_columns(
    smoothed_desired_position=pl.col('raw_desired_position').reverse().ewm_mean_by('timestamp', half_life=f'{smoothing_hl_secs}s').reverse()
)
desired_position_df

timestamp,symbol,expiry,dtte,rv_fair_annualized,rv_fair,rv_var,market_rv_fair_annualized,market_rv_fair,mean_iv_fair_annualized,mean_iv_fair,mean_iv_var,market_mean_iv_fair_annualized,market_mean_iv_fair,events_event_2_fair_annualized,events_event_2_fair,events_event_2_var,market_events_event_2_fair_annualized,market_events_event_2_fair,events_event_3_fair_annualized,events_event_3_fair,events_event_3_var,market_events_event_3_fair_annualized,market_events_event_3_fair,events_event_1_fair_annualized,events_event_1_fair,events_event_1_var,market_events_event_1_fair_annualized,market_events_event_1_fair,events_event_0_fair_annualized,events_event_0_fair,events_event_0_var,market_events_event_0_fair_annualized,market_events_event_0_fair,events_event_4_fair_annualized,events_event_4_fair,events_event_4_var,market_events_event_4_fair_annualized,market_events_event_4_fair,edge,var,raw_desired_position,smoothed_desired_position
datetime[μs],str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-01-01 00:00:00,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-21.029648,-6.6639e-7,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.49124,4.7255e-8,1.4176e-7,0.010906,3.4560e-10,0.0,0.0,0.0,0.0,0.0,-6.4660e-7,0.000001,-43841.000651,-122022.458545
2026-01-01 00:00:01,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-21.026143,-6.6628e-7,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.49124,4.7255e-8,1.4176e-7,0.010906,3.4560e-10,0.0,0.0,0.0,0.0,0.0,-6.4649e-7,0.000001,-43840.072907,-122082.694246
2026-01-01 00:00:02,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-21.022638,-6.6617e-7,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.49124,4.7255e-8,1.4176e-7,0.010906,3.4560e-10,0.0,0.0,0.0,0.0,0.0,-6.4638e-7,0.000001,-43839.144884,-122142.97707
2026-01-01 00:00:03,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-21.019133,-6.6606e-7,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.49124,4.7255e-8,1.4176e-7,0.010906,3.4560e-10,0.0,0.0,0.0,0.0,0.0,-6.4627e-7,0.000001,-43838.216581,-122203.307055
2026-01-01 00:00:04,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-21.015628,-6.6595e-7,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.49124,4.7255e-8,1.4176e-7,0.010906,3.4560e-10,0.0,0.0,0.0,0.0,0.0,-6.4615e-7,0.000001,-43837.287999,-122263.684238
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-01-01 23:59:56,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.7118e-8,3.2995e-10,-8.2189e6,-8.2189e6
2026-01-01 23:59:57,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.7118e-8,3.2995e-10,-8.2189e6,-8.2189e6
2026-01-01 23:59:58,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.010412,3.2995e-10,3.2995e-10,0.866202,2.7448e-8,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.7118e-8,3.2995e-10,-8.2189e6,-8.2189e6


# Sensechecking

In [12]:
### Sensechecking: Fair Value, Variance, Edge

def sum_series(series_list: list[pl.Series], length: int) -> pl.Series:
    total = pl.Series('total', [0.0] * length)
    for series in series_list:
        total = total + series
    return total

blocks = []
space_specs = {}

for stream in data_streams:
    for row in stream.snapshot.iter_rows(named=True):
        block = row['value_block']
        space_id = row['space_id']
        target_market_value = row['target_market_value']

        blocks.append(block)

        if space_id not in space_specs:
            space_specs[space_id] = {
                'target_market_value': target_market_value,
                'average_block_names': [],
                'offset_block_names': [],
                'all_block_names': [],
            }
        elif space_specs[space_id]['target_market_value'] != target_market_value:
            raise ValueError(
                f"Inconsistent target_market_value in space {space_id}: "
                f"{space_specs[space_id]['target_market_value']} vs {target_market_value}"
            )

        space_specs[space_id]['all_block_names'].append(block.block_name)
        if block.aggregation_logic == 'average':
            space_specs[space_id]['average_block_names'].append(block.block_name)
        else:
            space_specs[space_id]['offset_block_names'].append(block.block_name)

plot_df = fair_var_df.gather_every(60).drop_nulls(subset=['dtte'])
ts = plot_df['timestamp'].to_list()
space_ids = sorted(space_specs.keys(), key=lambda space_id: (space_id != 'shifting', space_id))
space_plot_data = {}

for space_id in space_ids:
    spec = space_specs[space_id]
    average_fair = sum_series(
        [plot_df[f'{block_name}_fair'] for block_name in spec['average_block_names']],
        len(plot_df),
    )
    if spec['average_block_names']:
        average_fair = average_fair / len(spec['average_block_names'])

    offset_fair = sum_series(
        [plot_df[f'{block_name}_fair'] for block_name in spec['offset_block_names']],
        len(plot_df),
    )
    average_market_fair = sum_series(
        [plot_df[f'market_{block_name}_fair'] for block_name in spec['average_block_names']],
        len(plot_df),
    )
    if spec['average_block_names']:
        average_market_fair = average_market_fair / len(spec['average_block_names'])

    offset_market_fair = sum_series(
        [plot_df[f'market_{block_name}_fair'] for block_name in spec['offset_block_names']],
        len(plot_df),
    )
    total_fair = average_fair + offset_fair
    total_market_fair = average_market_fair + offset_market_fair
    total_var = sum_series(
        [plot_df[f'{block_name}_var'] for block_name in spec['all_block_names']],
        len(plot_df),
    )

    space_plot_data[space_id] = {
        'average_fair': average_fair,
        'offset_fair': offset_fair,
        'total_fair': total_fair,
        'average_market_fair': average_market_fair,
        'offset_market_fair': offset_market_fair,
        'market_fair': total_market_fair,
        'total_var': total_var,
        'has_average': bool(spec['average_block_names']),
        'has_offset': bool(spec['offset_block_names']),
    }

In [13]:
# Graph 1: Fair Value by Stream vs Market Implied
fig1 = make_subplots(
    rows=len(space_ids),
    cols=1,
    shared_xaxes=True,
    subplot_titles=space_ids,
    vertical_spacing=0.04,
)

for row_idx, space_id in enumerate(space_ids, start=1):
    plot_spec = space_plot_data[space_id]

    fig1.add_trace(
        go.Scatter(
            x=ts,
            y=plot_spec['market_fair'].to_list(),
            name='market fair',
            legendgroup='market fair',
            showlegend=row_idx == 1,
            line=dict(dash='dash', color='rgba(255,255,255,0.6)', width=2),
        ),
        row=row_idx,
        col=1,
    )

    if plot_spec['has_average']:
        fig1.add_trace(
            go.Scatter(
                x=ts,
                y=plot_spec['average_fair'].to_list(),
                name='average fair',
                legendgroup='average fair',
                showlegend=row_idx == 1,
                line=dict(width=1, dash='dot', color='rgba(99,110,250,0.9)'),
            ),
            row=row_idx,
            col=1,
        )

    if plot_spec['has_offset']:
        fig1.add_trace(
            go.Scatter(
                x=ts,
                y=plot_spec['offset_fair'].to_list(),
                name='offset fair',
                legendgroup='offset fair',
                showlegend=row_idx == 1,
                line=dict(width=1, color='rgba(239,85,59,0.9)'),
            ),
            row=row_idx,
            col=1,
        )

    fig1.add_trace(
        go.Scatter(
            x=ts,
            y=plot_spec['total_fair'].to_list(),
            name='total fair',
            legendgroup='total fair',
            showlegend=row_idx == 1,
            line=dict(width=2, color='rgba(0,204,150,1)'),
        ),
        row=row_idx,
        col=1,
    )

fig1.update_layout(
    title='Fair Value by Space vs Market Fair',
    template='plotly_dark',
    height=260 * len(space_ids),
)
fig1.update_xaxes(title_text='Time', row=len(space_ids), col=1)
fig1.update_yaxes(title_text='Fair × Timestamp')
fig1.show()

In [14]:
# Graph 2: Stacked Variance Contribution by Stream
fig2 = go.Figure()

for space_id in space_ids:
    for block_name in space_specs[space_id]['all_block_names']:
        fig2.add_trace(
            go.Scatter(
                x=ts,
                y=plot_df[f'{block_name}_var'].to_list(),
                name=f'{space_id} | {block_name}',
                stackgroup='one',
                legendgroup=space_id,
            )
        )

fig2.update_layout(
    title='Variance Contribution by Block',
    template='plotly_dark',
    xaxis_title='Time',
    yaxis_title='Variance',
)
fig2.show()

In [15]:
# Graph 3: Edge by Space
fig3 = make_subplots(
    rows=len(space_ids),
    cols=1,
    shared_xaxes=True,
    subplot_titles=space_ids,
    vertical_spacing=0.04,
)

for row_idx, space_id in enumerate(space_ids, start=1):
    plot_spec = space_plot_data[space_id]
    edge_series = plot_spec['total_fair'] - plot_spec['market_fair']
    fig3.add_trace(
        go.Scatter(
            x=ts,
            y=edge_series.to_list(),
            name=f'{space_id} edge',
            showlegend=False,
        ),
        row=row_idx,
        col=1,
    )

fig3.update_layout(
    title='Edge by Space vs Market Fair',
    template='plotly_dark',
    height=260 * len(space_ids),
)
fig3.update_xaxes(title_text='Time', row=len(space_ids), col=1)
fig3.update_yaxes(title_text='Fair - Market Fair')
fig3.show()

In [16]:
# Graph 4: Raw and Smoothed Desired Position
pos_plot_df = desired_position_df.gather_every(60).drop_nulls(subset=['dtte'])
pos_ts = pos_plot_df['timestamp'].to_list()

fig4 = go.Figure()

fig4.add_trace(
    go.Scatter(
        x=pos_ts,
        y=pos_plot_df['raw_desired_position'].to_list(),
        name='Raw Desired Position',
        line=dict(width=1, color='rgba(99,110,250,0.4)'),
    )
)
fig4.add_trace(
    go.Scatter(
        x=pos_ts,
        y=pos_plot_df['smoothed_desired_position'].to_list(),
        name='Smoothed Desired Position',
        line=dict(width=2),
    )
)

fig4.update_layout(
    title='Raw and Smoothed Desired Position',
    template='plotly_dark',
    xaxis_title='Time',
    yaxis_title='Desired Position ($)',
)
fig4.show()